#  대화 이력 관리를 위한 메모리 구현(Chat History)

---

# 환경 설정 및 준비

`(1) Env 환경변수`

In [1]:
from dotenv import load_dotenv
load_dotenv()

True

`(2) 기본 라이브러리`

In [2]:
import os
from glob import glob

from pprint import pprint
import json

`(3) LLM 설정`

In [4]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model='gpt-4.1-mini',
    temperature=0.3,
    top_p=0.9, 
)

# **채팅 히스토리 관리**

- 채팅봇은 **이전 대화 내용을 문맥으로 활용**하여 자연스러운 대화 흐름을 유지함

- 기본 방식은 **이전 메시지를 프롬프트에 직접 포함**시키되, 오래된 메시지는 적절히 제거하여 정보량을 조절함

- 장시간 대화 시 **대화 내용 요약 저장** 등 고급 메모리 관리 기법을 활용함

### 1. **메시지 전달 방식(Message Passing)**

- **메시지 전달 방식**은 LangChain의 가장 기본적인 메모리 구현 방법으로, 이전 대화 기록을 체인에 직접 전달하여 문맥을 유지함

- **SystemMessage, HumanMessage, AIMessage** 등 다양한 메시지 유형을 ChatPromptTemplate으로 구조화하며, MessagesPlaceholder로 이전 대화 내용을 포함시킴

- 챗봇의 **기본 메모리 시스템 구현**에 활용되어 AI가 이전 대화 맥락을 이해하고 적절한 응답을 생성할 수 있게 함


In [5]:
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

# ChatPromptTemplate를 사용하여 챗봇의 초기 메시지를 정의
prompt = ChatPromptTemplate.from_messages([
    SystemMessage(content="You are a helpful assistant."),
    MessagesPlaceholder(variable_name="messages"),    # 메시지 목록을 동적으로 삽입하는 부분
])

# ChatPromptTemplate에 삽입할 메시지 목록을 정의
messages = [
        HumanMessage(content="안녕하세요. 제 이름은 홍길동입니다."),
        AIMessage(content="안녕하세요! 어떻게 도와드릴까요?"),
]    

# ChatPromptTemplate에 삽입할 메시지 목록을 업데이트하고 출력 
pprint(prompt.format(messages=messages))

('System: You are a helpful assistant.\n'
 'Human: 안녕하세요. 제 이름은 홍길동입니다.\n'
 'AI: 안녕하세요! 어떻게 도와드릴까요?')


In [ ]:
# 대화형 체인을 정의 (prompt -> llm)
chain = prompt | llm

# 기본적인 메시지 전달: 이전 메시지 목록에 새로운 메시지를 추가해서 전달
response = chain.invoke({
    "messages": messages + [HumanMessage(content="제 이름을 기억하나요?")] # 이전 메시지를 기억하는지 확인하는 질문 메시지 추가
})

pprint(response)

### 2. **RunnableWithMessageHistory**

- **RunnableWithMessageHistory**는 LangChain의 고급 대화 기록 관리 기능으로, 체인 실행 시 메시지 기록을 자동 저장하고 검색하는 래퍼 클래스임

- **get_session_history 콜백**을 통해 세션ID별 메시지 기록을 관리하며, Redis 등 외부 저장소와 통합하여 영구 보관도 가능함


`(1) 메모리 기반 로컬 저장소 활용`

- **InMemoryHistory 클래스**는 **BaseChatMessageHistory**와 BaseModel을 상속받아 메시지를 메모리에서 효율적으로 관리하며, messages 리스트로 BaseMessage 객체들을 순차 저장함

- **store 변수**는 전역 딕셔너리로 구현되어 session_id를 키로 각 세션의 InMemoryHistory 객체에 빠르게 접근하여 다중 사용자 환경에서 효율적 대화 관리를 지원함

- **get_session_history 함수**는 세션 관리의 진입점으로 팩토리 패턴을 구현하여 존재하지 않는 세션에 대해 자동으로 새 InMemoryHistory 객체를 생성함

In [ ]:
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.messages import BaseMessage
from pydantic import BaseModel, Field
from typing import List

# 메모리 기반 히스토리 구현
class InMemoryHistory(BaseChatMessageHistory, BaseModel):
    """메모리 히스토리 기본 클래스"""
    messages: List[BaseMessage] = Field(default_factory=list)
    
    def add_messages(self, messages: List[BaseMessage]) -> None:
        self.messages.extend(messages)
    
    def clear(self) -> None:
        self.messages = []

# 세션 저장소
store = {}

# 세션 ID로 히스토리 가져오기
def get_session_history(session_id: str) -> InMemoryHistory:
    if session_id not in store:
        store[session_id] = InMemoryHistory()
    return store[session_id]

In [ ]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables.history import RunnableWithMessageHistory

# 프롬프트 템플릿 설정
prompt = ChatPromptTemplate.from_messages([
    ("system", "당신은 여행 가이드입니다. 관광객에게 유용한 정보를 제공하세요."),
    MessagesPlaceholder(variable_name="history"), 
    ("human", "{input}")
])

# 프롬프트와 llm을 연결하여 체인 생성
chain = prompt | llm

# 히스토리 관리 추가  
chain_with_history = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="input",  # 사용자 입력 메시지 키
    history_messages_key="history"  # 히스토리 메시지 키
)

# 지정된 세션 ID(tourist_1)를 사용하여체인 실행
response = chain_with_history.invoke(
    {
        "input": "서울에서 가볼만한 곳을 추천해주세요."
    },
    config={"configurable": {"session_id": "tourist_1"}}
)


print(f"여행 가이드 답변:\n{response.content}")

In [ ]:
# 대화 히스토리 출력 
history = get_session_history("tourist_1")

pprint(history.messages)

In [ ]:
# 이전 대화 내용을 기반으로 새로운 질문을 추가하여 체인 실행
response = chain_with_history.invoke(
    {
        "input": "이전에 추천한 장소 중에서 가장 인기 있는 곳은 어디인가요?"
    },
    config={"configurable": {"session_id": "tourist_1"}}
)

print(f"여행 가이드 답변:\n{response.content}")

In [ ]:
# 대화 히스토리 출력
history = get_session_history("tourist_1")

pprint(history.messages)

`(2) SQLite 데이터베이스 영구 저장소 활용`

- **SQLite 통합 구현**을 위해 메시지 저장용 데이터베이스 테이블 구조를 정의하고, BaseChatMessageHistory를 상속받아 저장/조회 로직을 구현함

- **세션 ID 기준**으로 대화 내용을 구분 관리하며, 메시지 타입, 내용, 메타데이터, 타임스탬프 등을 체계적으로 저장함

- **효율적 검색과 활용**을 위해 적절한 인덱싱 전략 수립이 중요함

In [ ]:
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage
import sqlite3
from typing import List
import json

class SQLiteChatMessageHistory(BaseChatMessageHistory):
    """ 
    SQLite 데이터베이스를 사용하여 챗봇 대화 히스토리를 저장하는 클래스

    Attributes:
        session_id (str): 세션 ID
        db_path (str): SQLite 데이터베이스 파일 경로

    """
    def __init__(self, session_id: str, db_path: str = "chat_history.db"):
        self.session_id = session_id
        self.db_path = db_path
        self._create_tables()
    
    def _create_tables(self):
        """데이터베이스 테이블 생성"""
        conn = sqlite3.connect(self.db_path)
        cursor = conn.cursor()
        
        # 메시지 테이블 생성
        cursor.execute("""
            CREATE TABLE IF NOT EXISTS messages (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                session_id TEXT,
                message_type TEXT,
                content TEXT,
                metadata TEXT,
                created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
            )
        """)
        
        conn.commit()
        conn.close()
    
    def add_message(self, message: BaseMessage) -> None:
        """단일 메시지 추가"""
        conn = sqlite3.connect(self.db_path)
        cursor = conn.cursor()
        
        cursor.execute("""
            INSERT INTO messages (session_id, message_type, content, metadata)
            VALUES (?, ?, ?, ?)
        """, (
            self.session_id,
            message.__class__.__name__,
            message.content,
            json.dumps(message.additional_kwargs)
        ))
        
        conn.commit()
        conn.close()
    
    def add_messages(self, messages: List[BaseMessage]) -> None:
        """여러 메시지 추가"""
        conn = sqlite3.connect(self.db_path)
        cursor = conn.cursor()
        
        for message in messages:
            cursor.execute("""
                INSERT INTO messages (session_id, message_type, content, metadata)
                VALUES (?, ?, ?, ?)
            """, (
                self.session_id,
                message.__class__.__name__,
                message.content,
                json.dumps(message.additional_kwargs)
            ))
        
        conn.commit()
        conn.close()
    
    def clear(self) -> None:
        """세션의 모든 메시지 삭제"""
        conn = sqlite3.connect(self.db_path)
        cursor = conn.cursor()
        
        cursor.execute("""
            DELETE FROM messages WHERE session_id = ?
        """, (self.session_id,))
        
        conn.commit()
        conn.close()
    
    @property
    def messages(self) -> List[BaseMessage]:
        """저장된 메시지 조회"""
        conn = sqlite3.connect(self.db_path)
        cursor = conn.cursor()
        
        cursor.execute("""
            SELECT message_type, content, metadata
            FROM messages 
            WHERE session_id = ?
            ORDER BY created_at
        """, (self.session_id,))
        
        messages = []
        for row in cursor.fetchall():
            message_type, content, metadata = row
            if message_type == "HumanMessage":
                message = HumanMessage(content=content)
            else:
                message = AIMessage(content=content)
            
            if metadata:
                message.additional_kwargs = json.loads(metadata)
            
            messages.append(message)
        
        conn.close()
        return messages
    

# 세션 ID로 히스토리 가져오기
def get_chat_history(session_id: str) -> SQLiteChatMessageHistory:
    return SQLiteChatMessageHistory(session_id=session_id)

In [ ]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables.history import RunnableWithMessageHistory

# 프롬프트 템플릿 설정
prompt = ChatPromptTemplate.from_messages([
    ("system", "당신은 여행 가이드입니다. 관광객에게 유용한 정보를 제공하세요."),
    MessagesPlaceholder(variable_name="history"), 
    ("human", "{input}")
])

# 프롬프트와 llm을 연결하여 체인 생성
chain = prompt | llm

# 히스토리 관리 추가  
chain_with_history = RunnableWithMessageHistory(
    chain,
    get_chat_history,
    input_messages_key="input",
    history_messages_key="history"
)

# 지정된 세션 ID(tourist_1)를 사용하여체인 실행
response = chain_with_history.invoke(
    {
        "input": "수원에서 가볼만한 곳을 추천해주세요."
    },
    config={"configurable": {"session_id": "tourist_1"}}
)


print(f"여행 가이드 답변:\n{response.content}")

In [ ]:
# 대화 히스토리 출력
history = get_chat_history("tourist_1")

pprint(history.messages)

In [ ]:
# 이전 대화 내용을 기반으로 새로운 질문을 추가하여 체인 실행
response = chain_with_history.invoke(
    {
        "input": "이전에 추천한 장소 중에서 가장 인기 있는 곳은 어디인가요?"
    },
    config={"configurable": {"session_id": "tourist_1"}}
)

print(f"여행 가이드 답변:\n{response.content}")

In [ ]:
# 대화 히스토리 출력
history = get_chat_history("tourist_1")

pprint(history.messages)

In [ ]:
# 대화 히스토리 초기화
history.clear()

In [ ]:
# 대화 히스토리 출력
pprint(history.messages)

# **메시지 관리 기법**

- **대화 히스토리**가 일정 길이 초과 시 요약된 내용과 최근 메시지만으로 새로운 히스토리를 구성하여 컨텍스트 품질 유지와 토큰 사용량 최적화를 동시에 달성함

- **메시지 트리밍**은 오래된 대화 내역을 삭제하는 방식으로 컨텍스트 윈도우의 토큰 제한을 관리할 수 있음. 

- **장시간 대화** 시 이전 대화 내용을 한 문장으로 요약하여 컨텍스트로 활용함으로써 메모리 효율성을 높임


### 1. **메시지 트리밍(Message Trimming)**

* `trim_messages` 함수는 컨텍스트 윈도우의 토큰 제한을 관리하기 위한 핵심 도구로, 시스템 메시지를 포함할지 여부와 어디서부터 트리밍을 시작할지 등을 상세하게 설정할 수 있음

* 트리밍 전략으로 `last` 옵션을 사용하면 가장 최근의 메시지부터 시작하여 지정된 토큰 제한에 맞춰 이전 메시지들을 선택적으로 포함시킬 수 있음

In [ ]:
from langchain_core.messages import trim_messages

# 메시지 목록을 정의
orginal_messages = [
    HumanMessage(content="안녕하세요. 제 이름은 홍길동입니다."),
    AIMessage(content="안녕하세요! 어떻게 도와드릴까요?"),
    HumanMessage(content="제 이름을 기억하나요?")
]

# 트리머(trimmer) 생성
# 마지막 메시지 2개만 유지하도록 설정 (token은 메시지를 나타냄) 
trimmer = trim_messages(strategy="last", max_tokens=2, token_counter=len)

trimmed_messages = trimmer.invoke(orginal_messages)

pprint(trimmed_messages)

In [ ]:
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.messages import BaseMessage, trim_messages
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables.history import RunnableWithMessageHistory
from pydantic import BaseModel, Field
from typing import List

# 메시지 트리밍이 적용된 인메모리 히스토리 구현
class TrimmedInMemoryHistory(BaseChatMessageHistory, BaseModel):
    messages: List[BaseMessage] = Field(default_factory=list)
    max_tokens: int = Field(default=4)  # 유지할 최대 메시지 수
    
    def add_messages(self, messages: List[BaseMessage]) -> None:
        self.messages.extend(messages)
        # 메시지 추가 후 트리밍 수행
        trimmer = trim_messages(
            strategy="last",
            max_tokens=self.max_tokens,
            token_counter=len
        )
        self.messages = trimmer.invoke(self.messages)
    
    def clear(self) -> None:
        self.messages = []

# 세션 저장소
store = {}

# 세션 ID로 트리밍된 히스토리 가져오기
def get_trimmed_session_history(session_id: str) -> TrimmedInMemoryHistory:
    if session_id not in store:
        store[session_id] = TrimmedInMemoryHistory()
    return store[session_id]

# 프롬프트 템플릿 설정
prompt = ChatPromptTemplate.from_messages([
    ("system", "당신은 여행 가이드입니다. 관광객에게 유용한 정보를 제공하세요."),
    MessagesPlaceholder(variable_name="history"), 
    ("human", "{input}")
])

# 프롬프트와 llm을 연결하여 체인 생성
chain = prompt | llm

# 트리밍된 히스토리 관리 추가
chain_with_trimmed_history = RunnableWithMessageHistory(
    chain,
    get_trimmed_session_history,
    input_messages_key="input",
    history_messages_key="history"
)

# 지정된 세션 ID(tourist_1)를 사용하여 체인 실행
response = chain_with_trimmed_history.invoke(
    {
        "input": "서울에서 가볼만한 곳을 추천해주세요."
    },
    config={"configurable": {"session_id": "tourist_1"}}
)

print(f"여행 가이드 답변:\n{response.content}")

In [ ]:
# 대화 히스토리 출력
history = get_trimmed_session_history("tourist_1")

pprint(history.messages)

In [ ]:
# 이전 대화 내용을 기반으로 새로운 질문을 추가하여 체인 실행
response = chain_with_trimmed_history.invoke(
    {
        "input": "이전에 추천한 장소 중에서 가장 인기 있는 곳은 어디인가요?"
    },
    config={"configurable": {"session_id": "tourist_1"}}
)

print(f"여행 가이드 답변:\n{response.content}")

In [ ]:
# 대화 히스토리 출력
history = get_trimmed_session_history("tourist_1")

pprint(history.messages)

In [ ]:
# 이전 대화 내용을 기반으로 새로운 질문을 추가하여 체인 실행
response = chain_with_trimmed_history.invoke(
    {
        "input": "이 장소의 정식 명칭은 무엇인가요?"
    },
    config={"configurable": {"session_id": "tourist_1"}}
)

print(f"여행 가이드 답변:\n{response.content}")

In [ ]:
# 대화 히스토리 출력
history = get_trimmed_session_history("tourist_1")

pprint(history.messages)

### 2. **대화 요약 저장**

* 대화가 길어질 경우, 전체 대화 내용을 요약하여 컨텍스트로 활용하는 방식으로 이전 대화의 핵심을 추출

* 일반적으로 메시지 히스토리가 지정된 길이(예: 4개의 메시지)를 초과할 경우, 이전 대화들을 요약하고 가장 최근의 메시지만 유지하는 방식으로 새로운 대화 히스토리를 구성

* 이러한 요약 메모리 방식을 통해 토큰 사용량을 크게 줄이면서도 대화의 핵심 문맥을 유지할 수 있으며, 특히 장시간 진행되는 대화에서 효과적

In [ ]:
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.messages import BaseMessage, HumanMessage, SystemMessage
from langchain_openai import ChatOpenAI
from pydantic import BaseModel
from typing import List
    
class SummarizedInMemoryHistory(BaseChatMessageHistory, BaseModel):
    messages: List[BaseMessage] = Field(default_factory=list)
    summary_threshold: int = Field(default=6)  # 요약을 시작할 메시지 수
    # 요약에 사용할 LLM - lambda를 사용하여 필드가 초기화될 때마다 새로운 ChatOpenAI 인스턴스 생성
    llm: ChatOpenAI = Field(default_factory=lambda: ChatOpenAI(model="gpt-4.1-mini", temperature=0.1, top_p=0.9))  
    
    def add_messages(self, new_messages: List[BaseMessage]) -> None:
        self.messages.extend(new_messages)

        print(f"메시지 수: {len(self.messages)}")
        
        # 메시지 수가 임계값을 넘으면 요약 수행
        if len(self.messages) >= self.summary_threshold:
            # 마지막 사용자 메시지 저장 (HumanMessage, AIMessage 순서로 생성)
            last_user_message = self.messages[-2]
            last_ai_message = self.messages[-1]
            
            # 요약 생성
            summary_prompt = (
                "Distill the above chat messages into a single summary message. "
                "Include as many specific details as you can."
                "Use the original language and tone of the conversation."
            )
            
            summary_chain_messages = [
                SystemMessage(content=(
                    "You are a helpful assistant. "
                    "Your task is to summarize the conversation accurately."
                )),
                *self.messages[:-2],  # 마지막 대화 턴을 제외한 모든 메시지
                HumanMessage(content=summary_prompt)
            ]
            
            # 요약 생성
            summary = self.llm.invoke(summary_chain_messages)
            
            # 메시지 리스트 초기화 후 요약과 마지막 메시지 추가
            self.messages = [
                summary,
                last_user_message,
                last_ai_message
            ]
    
    def clear(self) -> None:
        self.messages = []

# 세션 저장소
store = {}

# 세션 ID로 요약된 히스토리 가져오기
def get_summarized_session_history(session_id: str) -> SummarizedInMemoryHistory:
    if session_id not in store:
        store[session_id] = SummarizedInMemoryHistory()
    return store[session_id]


# 프롬프트 템플릿 설정
prompt = ChatPromptTemplate.from_messages([
    ("system", "당신은 여행 가이드입니다. 관광객에게 유용한 정보를 제공하세요."),
    MessagesPlaceholder(variable_name="history"), 
    ("human", "{input}")
])

# 프롬프트와 llm을 연결하여 체인 생성
chain = prompt | llm

# 체인 구성
chain_with_summarized_history = RunnableWithMessageHistory(
    chain,
    get_summarized_session_history,
    input_messages_key="input",
    history_messages_key="history"
)

# 지정된 세션 ID(user_1)를 사용하여 체인 실행
response = chain_with_summarized_history.invoke(
    {
        "input": "서울에서 가볼만한 곳을 추천해주세요."
    },
    config={"configurable": {"session_id": "user_1"}}
)

print(f"여행 가이드 답변:\n{response.content}")

In [ ]:
# 대화 히스토리 출력
history = get_summarized_session_history("user_1")

pprint(history.messages)

In [ ]:
# 이전 대화 내용을 기반으로 새로운 질문을 추가하여 체인 실행
response = chain_with_summarized_history.invoke(
    {
        "input": "이전에 추천한 장소 중에서 가장 인기 있는 곳은 어디인가요?"
    },
    config={"configurable": {"session_id": "user_1"}}
)

print(f"여행 가이드 답변:\n{response.content}")

In [ ]:
# 대화 히스토리 출력
history = get_summarized_session_history("user_1")

pprint(history.messages)

In [ ]:
# 이전 대화 내용을 기반으로 새로운 질문을 추가하여 체인 실행
response = chain_with_summarized_history.invoke(
    {
        "input": "전주에서 가볼만한 곳을 추천해주세요."
    },
    config={"configurable": {"session_id": "user_1"}}
)

print(f"여행 가이드 답변:\n{response.content}")

In [ ]:
# 대화 히스토리 출력
history = get_summarized_session_history("user_1")

pprint(history.messages)

In [ ]:
# 이전 대화 내용을 기반으로 새로운 질문을 추가하여 체인 실행
response = chain_with_summarized_history.invoke(
    {
        "input": "두 도시에서 서로 비슷한 장소가 어디인가요?"
    },
    config={"configurable": {"session_id": "user_1"}}
)

print(f"여행 가이드 답변:\n{response.content}")

In [ ]:
# 대화 히스토리 출력
history = get_summarized_session_history("user_1")

pprint(history.messages)

In [ ]:
# 이전 대화 내용을 기반으로 새로운 질문을 추가하여 체인 실행
response = chain_with_summarized_history.invoke(
    {
        "input": "두 도시에서 각각 한 장소만 추천해주세요."
    },
    config={"configurable": {"session_id": "user_1"}}
)

print(f"여행 가이드 답변:\n{response.content}")

In [ ]:
# 대화 히스토리 출력
history = get_summarized_session_history("user_1")

pprint(history.messages)

In [ ]:
# 대화 히스토리 초기화
history.clear()

In [ ]:
# 대화 히스토리 출력
history = get_summarized_session_history("user_1")

pprint(history.messages)

---
# **[실습]**

- 메시지 트리밍과 대화 요약 저장을 결합하여 메시지를 관리하는 기능을 구현합니다. 
- 삭제된 메시지들을 요약하여 새로운 대화 히스토리를 생성하고, 이를 통해 토큰 사용량을 최적화합니다.

`1. 새로운 클래스 구조`

In [ ]:
class TrimmedAndSummarizedHistory(BaseChatMessageHistory, BaseModel):
    messages: List[BaseMessage]
    max_tokens: int  # 트리밍 기준 
    llm: ChatOpenAI
    summarized_messages: List[BaseMessage]  # 요약된 메시지 저장용

`2. 메시지 처리 로직`

- 새 메시지 추가시 max_tokens 체크
- 트리밍 발생하면 제거될 메시지 식별
- 제거 예정 메시지들은 요약하여 summarized_messages에 저장
- 현재 메시지는 트리밍된 상태로 유지

`3. 요약 프로세스`

- 트리밍으로 제거될 메시지들만 선별
- 선별된 메시지들에 대해 summary_chain 실행
- 요약본을 시스템 메시지로 변환하여 저장

In [ ]:
# 여기에 코드를 작성하세요. 